# 5. Stability and molecular packing
**Input:** raw simulation directory and explicit time window.
**Output:** disjoint block summaries, uncertainty sensitivity and species COM RDFs.
**Modules:** trajectory_analysis → stability. No cached reports are required.

These assess selected observables, not universal equilibrium. Temperature regulation alone is insufficient. Potential-energy/pressure logs and independent replicas remain additional checks.

Window selection uses **zero-based source dump frames**: `STARTFRAME` is inclusive and `ENDFRAME` exclusive. The defaults (`0`, `None`) use the entire loaded trajectory. With stride 2, `[3, 9)` selects source frames 4, 6, 8. These indices are not LAMMPS timesteps or rows after striding. Lag and diffusion-fit settings remain in ps.

Set `RUN_DIR` to your simulation folder. Leave `INPUT_FILE`, `DUMP_FILE`, and `DATA_FILE` as `None` for automatic discovery. Files directly in the folder take priority; subfolders are searched if necessary. If several files match, set the corresponding filename override. Keep units and other physical settings explicit.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations_with_replacement
from IPython.display import display
from MartiniFuncs.trajectory_analysis import analyze_run, molecular_window_statistics
from MartiniFuncs.stability import get_series_statistics, get_molecular_com_rdf

RUN_DIR = Path(r"/path/to/simulation")
INPUT_FILE = None  # optional filename override; None discovers automatically
DUMP_FILE = None
DATA_FILE = None
UNITS = "metal"
TIMESTEP_PS = 0.0005
TYPE_SYMBOLS = None
FRAME_STRIDE = 1
MAX_FRAMES = None
TEMPERATURE_DOF = "3N-3"  # unconstrained atoms, 3 COM DOF removed
# Select source frames [STARTFRAME, ENDFRAME), even with FRAME_STRIDE > 1.
STARTFRAME = 0     # inclusive source frame
ENDFRAME = None    # exclusive source frame; None uses the end
N_BLOCKS = 4
BATCH_DURATIONS_PS = [5., 10., 25., 50.]


### Mathematics: frames, time, units and molecular identity
Let $f$ be the zero-based source frame, $s_f$ its LAMMPS timestep, and $h=$ `TIMESTEP_PS` (ps per MD step). The displayed time is
$$t_f=s_fh.$$
`FRAME_INDEX` / `FRAME_INDICES` select $f$, not $s_f$. Where present, `FRAME_STRIDE` $q$ retains $f\bmod q=0$; `MAX_FRAMES` $F$ reads only $0\le f<F$. `STARTFRAME`, `ENDFRAME` or each `WINDOWS` entry $(a,b)$ keeps retained frames with $a\le f<b$; `None` means the available end. The lag spacing is $\Delta t=t_{j+1}-t_j$ between **retained** samples, not $h$. Paths and write/display switches do not enter these equations.

`UNITS` controls conversion on reading. In these workflows coordinates are Å, masses are atomic mass units, source forces are eV/Å, and reported velocities are Å/ps. `TYPE_SYMBOLS` sets the type-to-element mapping used to infer atomic properties when needed; an incorrect mapping can change masses and bond radii.

Where `BOND_SCALE` $s$ is used (default 1.3), atoms are connected when their periodic-image separation satisfies
$$\|\mathbf r_j+\mathbf nH-\mathbf r_i\|\le s(r_i^{\rm cov}+r_j^{\rm cov}).$$
$H$ contains cell vectors as rows, $\mathbf n$ is an integer image shift, and $r^{\rm cov}$ is the element's covalent radius (Å). Molecules are connected components of this inferred graph; increasing $s$ can merge close molecules. Persistent atom IDs are labels, not coordinate-array row indices.


### Mathematics: whole-frame density and kinetic temperature
For $N$ atoms, cell matrix $H$ (Å), masses $m_i$ (amu) and velocities $\mathbf v_i$:
$$V=|\det H|,\qquad \rho=1.66053906660\,\frac{\sum_i m_i}{V}\quad[\mathrm{g/cm^3}],$$
$$K=\tfrac12\sum_i m_i|\mathbf v_i|^2,\qquad T=\frac{2K}{f_{\rm DOF}k_B}.$$
Kinetic energy is evaluated in ASE's internal mass/velocity units, giving eV; $k_B$ is in eV/K. `TEMPERATURE_DOF` supplies $f_{\rm DOF}$; `"3N-3"` sets $f_{\rm DOF}=3N-3$ and `None` omits temperature. **Current notebook calls do not subtract COM velocity.** Choosing $3N-3$ changes only the denominator; it does not remove drift. A drift-corrected calculation instead uses $\mathbf v_i-\mathbf V$, where $\mathbf V=\sum_i m_i\mathbf v_i/\sum_i m_i$, and requires `remove_com=True`. Constraints require their own DOF accounting. Notebook 08 omits this derived temperature and can accept a documented thermo temperature instead.

A displayed whole-run mean is $\bar x=\sum_{j=0}^{T-1}x_j/T$; it averages saved frames, not independent samples.


In [ ]:
result = analyze_run(RUN_DIR, input_file=INPUT_FILE, dump_file=DUMP_FILE, data_file=DATA_FILE,
                     units=UNITS, timestep_ps=TIMESTEP_PS, type_symbols=TYPE_SYMBOLS,
                     frame_stride=FRAME_STRIDE, max_frames=MAX_FRAMES,
                     degrees_of_freedom=TEMPERATURE_DOF, progress=print)
print("Selected files:", result["files"])
window = molecular_window_statistics(result, startframe=STARTFRAME, endframe=ENDFRAME,
                                     max_lag_ps=TIMESTEP_PS)

print("Source frame range [start, end):", window["frame_window"])
frames = pd.DataFrame(result["frames"])
indices = np.flatnonzero(window["frame_mask"])
if N_BLOCKS < 2 or len(indices) < 3*N_BLOCKS:
    raise ValueError("Select at least three frames per block and at least two blocks")
blocks = np.array_split(indices, N_BLOCKS)
dt = np.diff(frames.time_ps.to_numpy()[indices])[0]
batch_sizes = tuple(sorted({max(2,int(round(t/dt))) for t in BATCH_DURATIONS_PS}))
print("Selected window (ps):", window["time_window_ps"])
display(pd.DataFrame([{"block":i,"first_ps":frames.time_ps.iloc[b[0]],"last_ps":frames.time_ps.iloc[b[-1]],
                      "frames":len(b)} for i,b in enumerate(blocks)]))


## Blocks and uncertainty
Standard deviations describe frame fluctuations. Batch-mean standard errors estimate uncertainty only for sufficiently decorrelated batches. Rising errors with larger batches can reveal slow variation; a few large batches give noisy errors. First-zero ACF truncation is heuristic and can miss oscillations or slow tails.

### Mathematics: scalar summaries and batch uncertainty
For $T$ samples $x_j$, compute $\bar x=T^{-1}\sum_jx_j$ and sample standard deviation $s_x=\sqrt{\sum_j(x_j-\bar x)^2/(T-1)}$. `N_BLOCKS` divides the selected samples into that many consecutive near-equal sections (earlier sections receive remainders). These section summaries differ from the smaller **batches** used to estimate errors.

`BATCH_DURATIONS_PS` $d$ is converted to $b=\max(2,\mathrm{round}(d/\Delta t))$ samples (NumPy/Python nearest-even rounding); only $B=\lfloor T/b\rfloor$ complete batches enter
$$\bar x_q=\frac1b\sum_{j=qb}^{(q+1)b-1}x_j,\qquad\mathrm{SE}(\bar x)=\frac{s_{\bar x_q}}{\sqrt B}.$$
The discarded tail is $T-Bb$ samples. The plotted batch duration is $b\Delta t$; the first-to-last sample span is $(b-1)\Delta t$. SE assumes batches are sufficiently independent. Check whether it stabilizes as batch size increases; a small SE alone is not proof of equilibrium.

`get_series_statistics` also uses a free-intercept least-squares scalar drift slope and empirical 5/50/95 percentiles. Its normalized ACF is
$$\rho_k=\frac{\sum_{j=0}^{T-k-1}(x_j-\bar x)(x_{j+k}-\bar x)}{(T-k)\,T^{-1}\sum_j(x_j-\bar x)^2},\quad
g=1+2\sum_{k=1}^{K-1}\rho_k,\quad N_{\rm eff}=T/g,\quad\tau_c=\Delta t\,g/2.$$
$K$ is the first nonpositive ACF lag (or the available end). These correlation estimates are heuristics, not equilibrium tests.


In [ ]:
columns = [k for k in ("temperature_K","density_g_cm3","volume_angstrom3") if k in frames]
rows = []
fig, axes = plt.subplots(len(columns), 2, figsize=(11,3*len(columns)), squeeze=False, layout="constrained")
for row_index,key in enumerate(columns):
    for i,b in enumerate(blocks):
        x = frames[key].to_numpy()[b]
        stats = get_series_statistics(x, sample_interval_ps=dt, batch_sizes=batch_sizes)
        rows.append({"observable":key,"block":i,"mean":stats["mean"],"frame_std":stats["std"]})
        axes[row_index,0].plot(frames.time_ps.iloc[b], x, lw=.7, label=f"block {i}")
    stats = get_series_statistics(frames[key].to_numpy()[indices], sample_interval_ps=dt, batch_sizes=batch_sizes)
    errors = stats["batch_mean_errors"]
    axes[row_index,1].plot([e["batch_duration_ps"] for e in errors],[e["standard_error"] for e in errors],"o-")
    axes[row_index,0].set(xlabel="Time (ps)",ylabel=key)
    axes[row_index,1].set(xlabel="Batch duration (ps)",ylabel="Estimated SE of mean")
display(pd.DataFrame(rows))
plt.show()


## Species COM packing
Use actual wrapped COMs and their instantaneous NPT box, not TOR trajectories. Self pairs are excluded. RDFs are normalized per frame using its volume and available pair count; radii stay below half the shortest box edge. Stable COM packing does not establish stable reactive orientations. Fully periodic orthorhombic cells are required.

### Mathematics: molecular RDF and nearest neighbors
Use instantaneous whole-molecule **wrapped** COMs, not accumulated TOR positions. For orthorhombic box lengths $\mathbf L_t$:
$$\mathbf d_{ij}=\mathbf R_j-\mathbf R_i-\mathbf L_t\left\lfloor(\mathbf R_j-\mathbf R_i)/\mathbf L_t+\tfrac12\right\rfloor.$$
For radial bin $[r_l,r_{l+1})$, let $n_{tl}$ count pair distances in that bin and $V_l=4\pi(r_{l+1}^3-r_l^3)/3$. Then
$$g_l=\frac1{T_s}\sum_{t\in\mathrm{sampled}}\frac{n_{tl}\prod_\alpha L_{t\alpha}}{P V_l},\quad
P=\begin{cases}M_A M_B&A\ne B,\\M_A(M_A-1)/2&A=B.\end{cases}$$
Like-species pairs are counted once, self-pairs excluded. Bin centers are $(r_l+r_{l+1})/2$. `RDF_BINS` sets the number of equal-width bins where configurable; notebook 05 fixes 64 bins and ends at $0.999\min_{t,\alpha}L_{t\alpha}/2$. Notebook 08 ends at half the shortest selected box edge, divides the window into four chronological sections, and samples every `STRUCTURE_STRIDE`-th retained frame **within each section**. These are descriptive curves without uncertainty bands.

Nearest-neighbor distances are $d_{ti}^{\rm nearest}=\min_j\|\mathbf d_{ij}\|$ (exclude self); the reported mean averages over central molecules and frames. Nearest distances are not limited by the RDF bin range.


In [ ]:
positions = result["wrapped_com_angstrom"]
if positions is None:
    print("RDF unavailable under the current cell convention:", result["notices"])
else:
    boxes = result["box_lengths_angstrom"]
    edges = np.linspace(0, boxes[indices].min()/2*.999, 65)
    nearest_rows = []
    for a,b in combinations_with_replacement(sorted(set(result["species"])),2):
        if a == b and np.sum(result["species"]==a)<2:
            continue
        fig, ax = plt.subplots(figsize=(8,4), layout="constrained")
        for i, block in enumerate(blocks):
            rdf = get_molecular_com_rdf(positions[block],boxes[block],result["species"],
                                        species_a=a,species_b=b,edges=edges)
            ax.plot(rdf["r_angstrom"],rdf["g_r"],label=f"block {i}")
            nearest_rows.append({"central":a,"neighbor":b,"block":i,
                                 "mean_nearest_angstrom":rdf["nearest_distances_angstrom"].mean()})
        ax.set(title=f"{a} / {b}",xlabel="COM separation (angstrom)",ylabel="g(r)")
        ax.legend()
        plt.show()
    display(pd.DataFrame(nearest_rows))
